In [ ]:
# Uncomment these in a fresh notebook environment.
#%pip install -r ../requirements.txt
#%pip install -e ..

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

GREEN_PHASE_COUNT = 4

def print_comparison_table(rows):
    columns = [
        "Policy",
        "Episodes",
        "Mean total reward",
        "Mean local queue",
        "Total completed steps",
        "Mean switches / episode",
    ]

    def format_value(value):
        if isinstance(value, float):
            return f"{value:.3f}"
        return str(value)

    widths = {column: len(column) for column in columns}
    for row in rows:
        for column in columns:
            widths[column] = max(widths[column], len(format_value(row.get(column, ""))))

    header = " | ".join(column.ljust(widths[column]) for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(separator)
    for row in rows:
        print(" | ".join(format_value(row.get(column, "")).ljust(widths[column]) for column in columns))


In [ ]:
from marl_tsc.simulation_generator import SimulationGenerator, DEFAULT_TRAFFIC_LIGHT_IDS
from marl_tsc.training import train_ppo, evaluate_policy, plot_training_histories
from marl_tsc.mappo import train_mappo
from marl_tsc.baselines import random_actions, fixed_time_actions


In [ ]:
#Constant parameters for the simulation and training
SIMULATION_DURATION = 4000
NUM_EPISODES = 30
EVALUATION_EPISODES = 10
TOTAL_TIMESTEPS = 200_000
MIN_GREEN_SECONDS = 10


## 1. Generate the SUMO simulation files

This default setup is a small `4x4` grid with four traffic-light agents. Change these values first when you want a different learning problem.


In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "main_flow.ipynb").exists() else NOTEBOOK_DIR
OUTPUT_DIR = PROJECT_ROOT / "outputs"

traffic_light_ids = list(DEFAULT_TRAFFIC_LIGHT_IDS)

generator = SimulationGenerator(
    output_dir=OUTPUT_DIR,
    grid_number=4,
    lane_number=2,
    traffic_light_ids=traffic_light_ids,
    trip_begin=0,
    trip_end=4000,
    trip_period=0.5,
    seed=42,
)

paths = generator.generate_all()
print("Generated:", paths.config_file)
print("Traffic-light agents:", traffic_light_ids)


Success.
calling /usr/share/sumo/bin/duarouter -n /home/isaac/Uni-Masters/MARL-TSC Group/marl-tsc/outputs/network.net.xml -r /home/isaac/Uni-Masters/MARL-TSC Group/marl-tsc/outputs/trips.trips.xml --ignore-errors --begin 0 --end 4000 --no-step-log --no-warnings -o /home/isaac/Uni-Masters/MARL-TSC Group/marl-tsc/outputs/routes.rou.xml
Success.
Generated: /home/isaac/Uni-Masters/MARL-TSC Group/marl-tsc/outputs/config.sumocfg
Traffic-light agents: ['B1', 'B2', 'C1', 'C2']


## 2. Run the simple baselines

Random and fixed-time are useful sanity checks before training. They use the same environment and the same evaluation seeds as the learned policies.


In [ ]:
random_result = evaluate_policy(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policy=random_actions,
    episodes=NUM_EPISODES,
    max_steps=SIMULATION_DURATION,
    seed=42,
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

fixed_time_result = evaluate_policy(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policy=fixed_time_actions,
    episodes=NUM_EPISODES,
    max_steps=SIMULATION_DURATION,
    seed=42,
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

baseline_rows = [
    {
        "Policy": "Random",
        "Episodes": random_result["episodes"],
        "Mean total reward": random_result["mean_total_reward"],
        "Mean local queue": random_result["mean_local_queue"],
        "Total completed steps": random_result["total_completed_steps"],
        "Mean switches / episode": random_result["mean_switches_per_episode"],
    },
    {
        "Policy": "Fixed time",
        "Episodes": fixed_time_result["episodes"],
        "Mean total reward": fixed_time_result["mean_total_reward"],
        "Mean local queue": fixed_time_result["mean_local_queue"],
        "Total completed steps": fixed_time_result["total_completed_steps"],
        "Mean switches / episode": fixed_time_result["mean_switches_per_episode"],
    },
]

print_comparison_table(baseline_rows)


TraCIException: Connection 'default' is already active.

## 3. Train

PPO uses parameter sharing through the PettingZoo/SuperSuit/SB3 wrapper. MAPPO adds a centralized critic and a shared decentralized actor.


In [ ]:
ppo_model, ppo_history, ppo_model_path = train_ppo(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    output_dir=OUTPUT_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    max_steps=SIMULATION_DURATION,
    seed=42,
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

mappo_model, mappo_history, mappo_model_path = train_mappo(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    output_dir=OUTPUT_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    rollout_steps=1024,
    max_steps=SIMULATION_DURATION,
    seed=42,
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

fig, ax = plot_training_histories(ppo_history + mappo_history)
plt.show()

print("Saved PPO model:", ppo_model_path)
print("Saved MAPPO model:", mappo_model_path)


## 4. Evaluate all policies on the same seeds

The final comparison table keeps the baseline runs and adds the shared PPO and MAPPO policies on the same evaluation seeds.


In [ ]:
shared_ppo_result = evaluate_policy(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policy=ppo_model,
    episodes=EVALUATION_EPISODES,
    max_steps=SIMULATION_DURATION,
    seed=42, 
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

shared_mappo_result = evaluate_policy(
    config_file=paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policy=mappo_model,
    episodes=EVALUATION_EPISODES,
    max_steps=SIMULATION_DURATION,
    seed=42, 
    env_kwargs={"green_phase_count": GREEN_PHASE_COUNT,
                "min_green_seconds": MIN_GREEN_SECONDS },
)

comparison_rows = baseline_rows + [
    {
        "Policy": "Shared PPO",
        "Episodes": shared_ppo_result["episodes"],
        "Mean total reward": shared_ppo_result["mean_total_reward"],
        "Mean local queue": shared_ppo_result["mean_local_queue"],
        "Total completed steps": shared_ppo_result["total_completed_steps"],
        "Mean switches / episode": shared_ppo_result["mean_switches_per_episode"],
    },
    {
        "Policy": "MAPPO",
        "Episodes": shared_mappo_result["episodes"],
        "Mean total reward": shared_mappo_result["mean_total_reward"],
        "Mean local queue": shared_mappo_result["mean_local_queue"],
        "Total completed steps": shared_mappo_result["total_completed_steps"],
        "Mean switches / episode": shared_mappo_result["mean_switches_per_episode"],
    },
]

print_comparison_table(comparison_rows)


## 5. Performance Analysis (100-Episode Moving Average)

This section visualizes the training progress by smoothing the rewards. It prioritizes the `moving_avg_episode_return` (available in MAPPO) and falls back to smoothing the `mean_training_reward` for PPO.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_moving_average_analysis(histories, window=100):
    if not histories:
        print("No history data found.")
        return

    df = pd.DataFrame(histories)
    fig, ax = plt.subplots(figsize=(12, 6))

    for algorithm in df['algorithm'].unique():
        algo_df = df[df['algorithm'] == algorithm].sort_values('timestep')
        
        # Identify the best metric to plot
        # MAPPO logs 'moving_avg_episode_return'; PPO logs 'mean_training_reward'
        metric = "moving_avg_episode_return" if "moving_avg_episode_return" in algo_df.columns else "mean_training_reward"
        
        # Calculate moving average (min_periods=1 allows plotting to start immediately)
        algo_df['smoothed'] = algo_df[metric].rolling(window=window, min_periods=1).mean()
        
        label_suffix = " (Episode Return)" if metric == "moving_avg_episode_return" else " (Step Reward)"
        line, = ax.plot(
            algo_df['timestep'], 
            algo_df['smoothed'], 
            label=f"{algorithm.upper()}{label_suffix}", 
            linewidth=2
        )
        
        # Plot raw data with high transparency to show variance/noise
        ax.plot(algo_df['timestep'], algo_df[metric], alpha=0.15, color=line.get_color())

    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Reward Value")
    ax.set_title(f"Training Performance: {window}-pt Moving Average")
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# Combine histories from both training runs and plot
combined_history = ppo_history + mappo_history
plot_moving_average_analysis(combined_history, window=100)

In [ ]:
len(mappo_history)

In [ ]:
len(ppo_history)